# 2 - Adjoint gradient, hockey-stick check, crack inversion

Hand-coded adjoint-state gradient = autodiff gradient (machine precision), verified with a Taylor / hockey-stick plot, then a single-crack L-BFGS inversion that prints a classic training loop.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/02_adjoint_fwi_hockey_stick.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    sys.path.insert(0, os.path.abspath("src"))  # make `import fwi` importable now

import torch
from IPython.display import Image, display
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)

## Adjoint kernel == autodiff (small grid, CPU float64)

In [ ]:
from fwi.problems import build_problem
from fwi.adjoint import adjoint_gradient
from fwi.misfit import autodiff_gradient, l2_misfit
from fwi.forward import forward
from fwi.gradient_test import taylor_test, clean_slope
from fwi import plotting

prob = build_problem('small', device='cpu', dtype=torch.float64)
kern, _ = adjoint_gradient(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg, active_mask=prob.active_mask)
g_ad, _ = autodiff_gradient(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg, active_mask=prob.active_mask)
print('adjoint vs autodiff relative-L2 =', float((kern-g_ad).norm()/g_ad.norm()))

## Hockey-stick (Taylor) verification

In [ ]:
def J_fn(m):
    with torch.no_grad():
        tr = forward(m, prob.src_sig, prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg).traces
    return float(l2_misfit(tr, prob.observed, prob.cfg.dt))
torch.manual_seed(0)
dm = torch.randn_like(kern) * prob.active_mask.to(kern.dtype)
hs = [10.0**k for k in range(6, -7, -1)]
hs, r1, r2 = taylor_test(J_fn, prob.start_alpha2, dm, kern, hs)
slope2, n = clean_slope(hs, r2)
print(f'second-order slope = {slope2:.3f} over {n} clean steps (expect ~2)')
display(Image(str(plotting.save_hockey_stick(hs, r1, r2, 'outputs/nb2_hockey.png',
    title=f'hockey-stick slope2={slope2:.2f}'))))

## Single-crack inversion with L-BFGS (classic training loop)
`verbose=True` prints per-iteration loss (J/J0) and grad norm, like a PyTorch training loop.

In [ ]:
from fwi.inversion import invert
probf = build_problem('crack', device=device, dtype=dtype)
alpha2_hat, history = invert(probf.start_alpha2, probf.observed, probf.src_sig,
    probf.src_i, probf.src_j, probf.rec_i, probf.rec_j, probf.cfg,
    active_mask=probf.active_mask, optimizer='lbfgs', n_iter=15, verbose=True)
display(Image(str(plotting.save_convergence(history, 'outputs/nb2_conv.png'))))
display(Image(str(plotting.save_field(alpha2_hat-probf.start_alpha2,
    'outputs/nb2_crack.png', title='recovered crack'))))